## Baseline-Modell

# *ALT* Analyse

## Bibliotheken und Daten laden

In [ ]:
# Notwendige Bibliotheken laden
import graphviz
import pm4py
from pm4py.objects.petri_net.utils import initial_marking
from pm4py.visualization.transition_system.util.visualize_graphviz import visualize

# Mining
from pm4py.algo.discovery.alpha import algorithm as alpha_miner
from pm4py.algo.discovery.heuristics import algorithm as heuristic_miner
from pm4py.algo.discovery.inductive import algorithm as inductive_miner
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
from pm4py.statistics.start_activities.log import get as start_activities
from pm4py.statistics.end_activities.log import get as end_activities
from pm4py.visualization.dfg import visualizer as dfg_visualization
from pm4py.algo.discovery.ilp import algorithm as ilp_miner
from pm4py.algo.discovery.genetic import algorithm as genetic_miner

# Evaluierung
from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness
from pm4py.algo.evaluation.precision import algorithm as precision_evaluator
from pm4py.algo.evaluation.generalization import algorithm as generalization_evaluator
from pm4py.algo.evaluation.simplicity import algorithm as simplicity_evaluator
from pm4py.objects.conversion.dfg import converter as dfg_converter
import pandas as pd

# Daten laden
# Pfad bitte anpassen, falls ihr es bei euch laufen lasst.
df = pm4py.read_xes('/Users/elias/Desktop/Uni/4. Semester : Masterarbeit/2026-04-06 TYT_event_log V2.0.xes')

In [ ]:
df.head()

In [ ]:
print(df.isna().sum())

In [ ]:
log = df.copy()

## Mining

In [ ]:
# Log komprimieren
from pm4py.statistics.variants.log import get as variants_get

variants = variants_get.get_variants(log)

variant_freq = {variant: len(traces) for variant, traces in variants.items()}



from pm4py.objects.log.obj import EventLog, Trace, Event

compressed_log = EventLog()

for variant, freq in variant_freq.items():
    trace = Trace()

    # Gewicht speichern
    trace.attributes["weight"] = freq

    for act in variant:
        event = Event({"concept:name": act})
        trace.append(event)

    compressed_log.append(trace)



for i, trace in enumerate(compressed_log):
    print(f"Trace {i}")
    print("Gewicht:", trace.attributes.get("weight", 1))

    activities = [event["concept:name"] for event in trace]
    print("Ablauf:", " -> ".join(activities))
    print("-" * 40)

In [ ]:
# Mining

# Abkürzungen

# im = initial_marking
# fm = final_marking
# alpha_m = Alpha Miner
# heuristic_m = Heuristic Miner
# inductive_m = Inductive Miner
# ilp_m = ILP Miner
# dfg_m = DFG Miner
# genetic_m = Genetic Miner

In [ ]:
# Alpha Miner

net_alpha_m, im_alpha_m, fm_alpha_m = alpha_miner.apply(log)
pm4py.view_petri_net(net_alpha_m, im_alpha_m, fm_alpha_m)

In [ ]:
# Heuristic Miner

net_heuristic_m, im_heuristic_m, fm_heuristic_m = heuristic_miner.apply(log)
pm4py.view_petri_net(net_heuristic_m, im_heuristic_m, fm_heuristic_m)

In [ ]:
# Inductive Miner

tree = inductive_miner.apply(log)
net_inductive_m, im_inductive_m, fm_inductive_m = pm4py.convert_to_petri_net(tree)
pm4py.view_petri_net(net_inductive_m, im_inductive_m, fm_inductive_m)

In [ ]:
# DFG (Directly-Follows-Graph) anstelle von Fuzzy Miner als Approximation
# PM4Py hat keinen Fuzzy Miner, höchstens ProM

# DFG (Directly Follows Graph) mit Häufigkeiten/Durchlaufzeiten und unterschiedlicher Pfeildicke

dfg = dfg_discovery.apply(log)

# Visualisierung mit Häufigkeiten
start_acts = start_activities.get_start_activities(log)
end_acts = end_activities.get_end_activities(log)

gviz = dfg_visualization.apply(
    dfg, log=log, variant=dfg_visualization.Variants.FREQUENCY, parameters={"start_activities": start_acts, "end_activities": end_acts}
)

dfg_visualization.view(gviz)

In [ ]:
# ILP Miner

# lief 22min und es ging nicht voran

# PuLP NICHT installieren, dann geht der Codeblock nicht mehr!!
'''
net_ilp_m, im_ilp_m, fm_ilp_m = ilp_miner.apply(log)
pm4py.view_petri_net(net_ilp_m, im_ilp_m, fm_ilp_m)
'''

In [ ]:
# Genetic Miner
'''
net_genetic_m, im_genetic_m, fm_genetic_m = genetic_miner.apply(log)
pm4py.view_petri_net(net_genetic_m, im_genetic_m, fm_genetic_m)
'''

## Evaluierung
Fitness, Precision, Generalisation und Simplicity

In [ ]:
# Evaluierung als Definition, um das später leichter zu haben.
# Wählt eigenständig den Token Based Replay oder Alignment für Fitness für den jeweiligen Miner!

def evaluate_model(log, net, im, fm):
    fitness_result = replay_fitness.apply(log, net, im, fm)
    precision = precision_evaluator.apply(log, net, im, fm)
    generalization = generalization_evaluator.apply(log, net, im, fm)
    simplicity = simplicity_evaluator.apply(net)

    fitness_value = fitness_result.get('log_fitness') or fitness_result.get('average_trace_fitness')

    print(f"Fitness: {fitness_result}"),
    print(f"Precision: {precision * 100:.4f}%"),
    print(f"Generalization: {generalization :.4f}"),
    print(f"Simplicity: {simplicity :.4f}")

    # Rückgabe als Dictionary für Tabelle
    return {
        "Fitness": fitness_value,
        "Precision": precision,
        "Generalization": generalization,
        "Simplicity": simplicity
    }

# Dictionary zum Speichern der Ergebnisse
all_results = {}

In [ ]:
# Evaluierung Alpha Miner
print(f'Alpha Miner')
all_results['Alpha'] = evaluate_model(log, net_alpha_m, im_alpha_m, fm_alpha_m)

In [ ]:
# Evaluierung Heuristic Miner
print(f'Heuristic Miner')
all_results['Heuristic'] = evaluate_model(log, net_heuristic_m, im_heuristic_m, fm_heuristic_m)

In [ ]:
# Evaluierung Inductive Miner
print(f'Inductive Miner')
all_results['Inductive'] = evaluate_model(log, net_inductive_m, im_inductive_m, fm_inductive_m)

In [ ]:
# Evaluierung DFG
net_dfg, im_dfg, fm_dfg = dfg_converter.apply(dfg)

print("DFG (converted)")
all_results['DFG (converted)'] = evaluate_model(log, net_dfg, im_dfg, fm_dfg)

In [ ]:
# Evaluierung ILP Miner
'''print(f'ILP Miner')
all_results['ILP'] = evaluate_model(log, net_ilp_m, im_ilp_m, fm_ilp_m)'''

In [ ]:
# Evaluierung Genetic Miner
'''print(f'Genetic Miner')
all_results['Genetic'] = evaluate_model(log, net_genetic_m, im_genetic_m, fm_genetic_m)'''

## Vergleich

In [ ]:
# Pandas DataFrames aus Dictionary erstellen
df_comparison = pd.DataFrame.from_dict(all_results, orient='index')

# Formatierung für die Anzeige (% und :.4f)
df_display = df_comparison.copy()
for col in ["Fitness", "Precision"]:
    df_display[col] = df_display[col].apply(lambda x: f"{x * 100:.2f}%")

for col in ["Generalization", "Simplicity"]:
    df_display[col] = df_display[col].apply(lambda x: f"{x:.4f}")

print("Übersicht:")
print(df_display)
# oder display(df_display)

# *NEU* Analyse

## Bibliothek und Daten laden

In [ ]:
# Mining
import pm4py

# Evaluierung
from pm4py.algo.evaluation.replay_fitness import algorithm as replay_fitness
from pm4py.algo.evaluation.precision import algorithm as precision_evaluator
from pm4py.algo.evaluation.generalization import algorithm as generalization_evaluator
from pm4py.algo.evaluation.simplicity import algorithm as simplicity_evaluator
from pm4py.objects.conversion.dfg import converter as dfg_converter
from pm4py.visualization.petri_net import visualizer as pn_visualizer
import pandas as pd

# Daten
#df = pm4py.read_xes('/Users/elias/Desktop/Uni/4. Semester : Masterarbeit/2026-04-06 TYT_event_log V2.0 daily.xes')
df = pm4py.read_xes('/Users/elias/Desktop/Uni/4. Semester : Masterarbeit/2026-04-06 TYT_event_log V2.0.xes')
log = pm4py.convert_to_event_log(df)

## Mining

In [ ]:
# Abkürzungen

# im = initial_marking
# fm = final_marking
# alpha_m = Alpha Miner
# heuristic_m = Heuristic Miner
# inductive_m = Inductive Miner
# ilp_m = ILP Miner
# dfg_m = DFG Miner
# genetic_m = Genetic Miner

In [ ]:
# Alpha Miner mit Häufigkeiten
gviz = pn_visualizer.apply(
    net_alpha_m,
    im_alpha_m,
    fm_alpha_m,
    log=log,
    variant=pn_visualizer.Variants.FREQUENCY
)

pn_visualizer.view(gviz)

In [ ]:
# Alpha Miner
net_alpha_m, im_alpha_m, fm_alpha_m = pm4py.discover_petri_net_alpha(log)
pm4py.view_petri_net(net_alpha_m, im_alpha_m, fm_alpha_m)

In [ ]:
# Heuristic Miner mit Häufigkeiten
gviz = pn_visualizer.apply(
    net_heuristic_m,
    im_heuristic_m,
    fm_heuristic_m,
    log=log,
    variant=pn_visualizer.Variants.FREQUENCY
)

pn_visualizer.view(gviz)

In [ ]:
# Heuristic Miner
net_heuristic_m, im_heuristic_m, fm_heuristic_m = pm4py.discover_petri_net_heuristics(log)
pm4py.view_petri_net(net_heuristic_m, im_heuristic_m, fm_heuristic_m)

In [ ]:
# Inductive Miner mit Häufigkeiten
gviz = pn_visualizer.apply(
    net_inductive_m,
    im_inductive_m,
    fm_inductive_m,
    log=log,
    variant=pn_visualizer.Variants.FREQUENCY
)

pn_visualizer.view(gviz)

In [ ]:
# Inductive Miner
net_inductive_m, im_inductive_m, fm_inductive_m = pm4py.discover_petri_net_inductive(log)
pm4py.view_petri_net(net_inductive_m, im_inductive_m, fm_inductive_m)

In [ ]:
'''
# Inductive Miner mit tree Zwischenschritt (falls man den später mal brauchen sollte)
tree = pm4py.discover_process_tree_inductive(log)
net_inductive_m, im_inductive_m, fm_inductive_m = pm4py.convert_to_petri_net(tree)
pm4py.view_petri_net(net_inductive_m, im_inductive_m, fm_inductive_m)
'''

In [ ]:
# DFG Miner
dfg_graph, start_activities, end_activities = pm4py.discover_dfg(log)
pm4py.view_dfg(dfg_graph, start_activities, end_activities)

In [ ]:
'''
# ILP Miner
net_ilp_m, im_ilp_m, fm_ilp_m = pm4py.discover_petri_net_ilp(log)
pm4py.view_petri_net(net_ilp_m, im_ilp_m, fm_ilp_m)
'''

In [ ]:
'''
# Genetic Miner
net_genetic_m, im_genetic_m, fm_genetic_m = pm4py.discover_petri_net_genetic(log)
pm4py.view_petri_net(net_genetic_m, im_genetic_m, fm_genetic_m)
'''

## Evaluierung
Fitness, Precision, Generalisation und Simplicity

In [ ]:
# Evaluierung als Definition, um das später leichter zu haben.
# Token Based Replay gewählt, da es deutlich schneller geht.

def evaluate_model(log, net, im, fm):
    fitness = replay_fitness.apply(log, net, im, fm, variant=replay_fitness.Variants.TOKEN_BASED)
    precision = precision_evaluator.apply(log, net, im, fm, variant=precision_evaluator.Variants.ETCONFORMANCE_TOKEN)
    generalization = generalization_evaluator.apply(log, net, im, fm, variant=generalization_evaluator.Variants.GENERALIZATION_TOKEN)
    simplicity = simplicity_evaluator.apply(net)

    fitness_value = fitness.get('log_fitness') or fitness.get('average_trace_fitness')

    print(f"Fitness: {fitness}"),
    print(f"Precision: {precision * 100:.4f}%"),
    print(f"Generalization: {generalization :.4f}"),
    print(f"Simplicity: {simplicity :.4f}")

    # Rückgabe als Dictionary für Tabelle
    return {
        "Fitness": fitness_value,
        "Precision": precision,
        "Generalization": generalization,
        "Simplicity": simplicity
    }

# Dictionary zum Speichern der Ergebnisse
all_results = {}

In [ ]:
# Evaluierung Alpha Miner
print(f'Alpha Miner')
all_results['Alpha'] = evaluate_model(log, net_alpha_m, im_alpha_m, fm_alpha_m)

In [ ]:
# Evaluierung Heuristic Miner
print(f'Heuristic Miner')
all_results['Heuristic'] = evaluate_model(log, net_heuristic_m, im_heuristic_m, fm_heuristic_m)

In [ ]:
# Evaluierung Inductive Miner
print(f'Inductive Miner')
all_results['Inductive'] = evaluate_model(log, net_inductive_m, im_inductive_m, fm_inductive_m)

In [ ]:
# schnell
# Evaluierung DFG
from pm4py.algo.discovery.dfg import algorithm as dfg_discovery
dfg = dfg_discovery.apply(log)
net_dfg, im_dfg, fm_dfg = dfg_converter.apply(dfg)

print("DFG (converted)")
all_results['DFG (converted)'] = evaluate_model(log, net_dfg, im_dfg, fm_dfg)

In [ ]:
'''
# langsam
# Evaluierung DFG
# Wir erstellen das für den Konverter notwendige Input-Dictionary
dfg_dict = {
    "dfg": dfg_graph,
    "start_activities": start_activities,
    "end_activities": end_activities
}

# Jetzt rufen wir den Konverter auf
# Der Konverter erwartet in der Regel nur das DFG-Objekt als primäres Argument
# oder eine spezifische Variante zur Umwandlung.
net_dfg, im_dfg, fm_dfg = dfg_converter.apply(dfg_dict)

print("DFG (converted)")
all_results['DFG (converted)'] = evaluate_model(log, net_dfg, im_dfg, fm_dfg)
'''

In [ ]:
# Evaluierung ILP Miner
'''print(f'ILP Miner')
all_results['ILP'] = evaluate_model(log, net_ilp_m, im_ilp_m, fm_ilp_m)'''

In [ ]:
# Evaluierung Genetic Miner
'''print(f'Genetic Miner')
all_results['Genetic'] = evaluate_model(log, net_genetic_m, im_genetic_m, fm_genetic_m)'''

## Vergleich

In [ ]:
# Pandas DataFrames aus Dictionary erstellen
df_comparison = pd.DataFrame.from_dict(all_results, orient='index')

# Formatierung für die Anzeige (% und :.4f)
df_display = df_comparison.copy()
for col in ["Fitness", "Precision"]:
    df_display[col] = df_display[col].apply(lambda x: f"{x * 100:.2f}%")

for col in ["Generalization", "Simplicity"]:
    df_display[col] = df_display[col].apply(lambda x: f"{x:.4f}")

print("Übersicht:")
print(df_display)
# oder display(df_display)